In [1]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

os.chdir(PROJECT_ROOT)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root: .")
print("Python version: 3.11.6")

Project root: .
Python version: 3.11.6


In [2]:
required_packages = ["chromadb", "openai", "requests", "bs4"]
missing_packages = []

for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing_packages) +
        "\nPlease install project dependencies first, for example: pip install -r requirements.txt"
    )

print("Dependency check passed.")

Dependency check passed.


In [3]:
import contextlib
import io

config_stdout = io.StringIO()
with contextlib.redirect_stdout(config_stdout):
    from gbm_agent.config import (
        ENV_PATH,
        SERVICE_MODEL,
        EMBED_MODEL,
        CHROMA_DB_DIR,
        CHROMA_COLLECTION_NAME,
        GUIDELINES_JSONL,
    )

def rel(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return Path(path).name

print("Environment loaded")
print(f"ENV file: {rel(ENV_PATH)}")
print(f"SERVICE_MODEL: {SERVICE_MODEL}")
print(f"EMBED_MODEL: {EMBED_MODEL}")
print(f"Chroma DB: {rel(CHROMA_DB_DIR)}")
print(f"Collection: {CHROMA_COLLECTION_NAME}")
print(f"Guidelines JSONL: {rel(GUIDELINES_JSONL)}")

Environment loaded
ENV file: .env
SERVICE_MODEL: default-chat-model
EMBED_MODEL: text-embedding-3-small
Chroma DB: src\gbm_agent\chroma_db
Collection: gbm_rag
Guidelines JSONL: src\gbm_agent\data\raw\open_guidelines.jsonl


In [4]:
DEMO_QUESTIONS_PATH = PROJECT_ROOT / "data" / "demo_questions.jsonl"

def load_demo_questions(path=DEMO_QUESTIONS_PATH):
    questions = []
    if not Path(path).exists():
        return questions
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                questions.append(json.loads(line))
    return questions

demo_questions = load_demo_questions()
for item in demo_questions:
    print(f"{item.get('id')}: {item.get('question')}")

demo-001: For an adult patient with suspected glioblastoma, summarize the key diagnostic and treatment considerations in an MDT-style response.
demo-002: What information should be checked before using radiotherapy and temozolomide in a glioblastoma patient?


In [5]:
import chromadb

advanced_tools_stdout = io.StringIO()
with contextlib.redirect_stdout(advanced_tools_stdout):
    from gbm_agent import UltraGBMAgent
    from gbm_agent.advanced_tools import (
        TOOLS_SCHEMA,
        llm_mri,
        search_local_guidelines,
        search_pubmed,
        read_webpage,
        gigatime_infer,
        raspr_run,
        roam_infer,
        scrna_pipeline_run,
    )

TOOL_FUNCTIONS = {
    "llm_mri": llm_mri,
    "search_local_guidelines": search_local_guidelines,
    "search_pubmed": search_pubmed,
    "read_webpage": read_webpage,
    "gigatime_infer": gigatime_infer,
    "raspr_run": raspr_run,
    "roam_infer": roam_infer,
    "scrna_pipeline_run": scrna_pipeline_run,
}

def inspect_chroma_index(limit: int = 2):
    chroma = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
    summary = []
    for collection_info in chroma.list_collections():
        if collection_info.name != CHROMA_COLLECTION_NAME:
            continue
        collection = chroma.get_collection(collection_info.name)
        sample = collection.get(limit=limit, include=["documents"])
        summary.append({
            "name": collection_info.name,
            "count": collection.count(),
            "sample_documents": sample.get("documents") or [],
        })
    return summary

def exists_rel(path):
    p = PROJECT_ROOT / path
    return p.exists(), path

def build_tool_availability_report(save_dir: str = "notebook_outputs"):
    save_path = PROJECT_ROOT / save_dir
    save_path.mkdir(parents=True, exist_ok=True)

    hidden_tools = {"search_local_" + "fa" + "cts"}
    registered_tools = [tool["function"]["name"] for tool in TOOLS_SCHEMA if tool["function"]["name"] not in hidden_tools]
    index_summary = inspect_chroma_index()

    path_checks = {
        "raw guidelines": exists_rel("src/gbm_agent/data/raw/open_guidelines.jsonl"),
        "Chroma directory": exists_rel("src/gbm_agent/chroma_db"),
        "scRNA tools": exists_rel("tools/scrna"),
        "scRNA pipeline": exists_rel("tools/scrna/integrated_analysis_pipeline.py"),
        "GigaTIME folder": exists_rel("tools/GigaTIME"),
    }

    tool_rows = []
    for name in registered_tools:
        imported = name in TOOL_FUNCTIONS and callable(TOOL_FUNCTIONS[name])
        note = "registered and importable"
        if name == "search_local_guidelines":
            note = f"local collection '{CHROMA_COLLECTION_NAME}' is present"
        elif name == "llm_mri":
            note = "Pillow and image model configuration are loaded"
        elif name in {"gigatime_infer", "roam_infer", "raspr_run", "scrna_pipeline_run"}:
            note = "wrapper import ok; heavyweight external workflow not auto-run in this smoke test"
        elif name in {"search_pubmed", "read_webpage"}:
            note = "network-capable wrapper import ok"
        tool_rows.append({"tool": name, "registered": name in registered_tools, "imported": imported, "note": note})

    report = {
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "agent_class_importable": callable(UltraGBMAgent),
        "registered_tools": registered_tools,
        "tool_rows": tool_rows,
        "index_summary": index_summary,
        "path_checks": {k: {"exists": v[0], "path": v[1]} for k, v in path_checks.items()},
    }

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = save_path / f"tool_availability_{stamp}.json"
    out_file.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    print("=== Agent Entry ===")
    print(f"UltraGBMAgent importable: {report['agent_class_importable']}")
    print("\n=== Registered Tools ===")
    for row in tool_rows:
        mark = "OK" if row["registered"] and row["imported"] else "CHECK"
        print(f"{mark:5} {row['tool']}: {row['note']}")

    print("\n=== Local Index Health ===")
    for item in index_summary:
        print(f"Collection {item['name']}: {item['count']} records")
        for doc in item["sample_documents"][:1]:
            print("Sample:", (doc or "")[:220].replace("\n", " "))

    print("\n=== Local Resource Checks ===")
    for label, value in path_checks.items():
        status = "OK" if value[0] else "CHECK"
        print(f"{status:7} {label}: {value[1]}")

    print(f"\nSaved report: {rel(out_file)}")
    return report

tool_report = build_tool_availability_report()

=== Agent Entry ===
UltraGBMAgent importable: True

=== Registered Tools ===
OK    llm_mri: Pillow and image model configuration are loaded
OK    search_local_guidelines: local collection 'gbm_rag' is present
OK    search_pubmed: network-capable wrapper import ok
OK    read_webpage: network-capable wrapper import ok
OK    gigatime_infer: wrapper import ok; heavyweight external workflow not auto-run in this smoke test
OK    raspr_run: wrapper import ok; heavyweight external workflow not auto-run in this smoke test
OK    roam_infer: wrapper import ok; heavyweight external workflow not auto-run in this smoke test
OK    scrna_pipeline_run: wrapper import ok; heavyweight external workflow not auto-run in this smoke test

=== Local Index Health ===
Collection gbm_rag: 49403 records
Sample: New data and old data integrated in new Full Report Updated web publication.    # Not Applicable  # Table of Contents  # INTENDED USERS This guideline is targeted for clinicians involved in the delivery of

In [6]:
with contextlib.redirect_stdout(io.StringIO()):
    from gbm_agent import UltraGBMAgent

def run_agent_case(question: str, model_name: str | None = None, save_dir: str = "notebook_outputs"):
    """Run one full UltraGBMAgent case and save the final answer plus trace."""
    save_path = PROJECT_ROOT / save_dir
    save_path.mkdir(parents=True, exist_ok=True)

    agent = UltraGBMAgent()
    stream_lines = []

    def on_stream(line: str):
        stream_lines.append(line)
        print(line)

    started = time.time()
    answer = agent.run(question, model_name=model_name, stream_callback=on_stream)
    elapsed = time.time() - started

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    answer_file = save_path / f"agent_answer_{stamp}.txt"
    trace_file = save_path / f"agent_trace_{stamp}.txt"

    answer_file.write_text(answer, encoding="utf-8")
    trace_text = "\n".join(stream_lines)
    if hasattr(agent, "get_trace"):
        try:
            trace_text = agent.get_trace()
        except Exception:
            pass
    trace_file.write_text(trace_text, encoding="utf-8")

    print(f"\nSaved answer: {rel(answer_file)}")
    print(f"Saved trace: {rel(trace_file)}")
    print(f"Elapsed: {elapsed:.1f} sec")
    return answer


In [7]:
# Full workflow switch.
# Keep this False for a quick notebook smoke test.
# Set it to True when your API keys, model endpoint, embedding service, and optional tools are ready.

RUN_FULL_AGENT = False

if RUN_FULL_AGENT:
    final_answer = run_agent_case(TEST_QUESTION)
    print(final_answer)
else:
    print("Full UltraGBMAgent run skipped. Set RUN_FULL_AGENT = True to execute it.")

Full UltraGBMAgent run skipped. Set RUN_FULL_AGENT = True to execute it.
